In [ ]:
pip install pandas

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("Dataset_14-day_AA_depression_symptoms_mood_and_PHQ-9.csv")

df.head()

,Unnamed: 0,user_id,phq1,phq2,phq3,phq4,phq5,phq6,phq7,phq8,...,q14,q16,q46,q47,happiness.score,time,period.name,start.time,phq.day,id
0,1,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,0.0,NaN,NaN,NaN,2,2017-01-22 20:11:59,evening,2017-01-09 07:22:37,-0.951921,1
1,2,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,NaN,NaN,1.0,NaN,3,2017-02-08 22:53:06,evening,2017-01-09 07:22:37,16.159965,1
2,3,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,NaN,NaN,NaN,NaN,3,2017-02-08 08:00:46,morning,2017-01-09 07:22:37,15.540289,1
3,4,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,NaN,1.0,NaN,NaN,2,2017-01-22 14:01:25,midday,2017-01-09 07:22:37,-1.209259,1
4,5,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,1.0,NaN,1.0,NaN,2,2017-01-21 15:37:24,midday,2017-01-09 07:22:37,-2.142604,1


In [3]:
# 3. Check dataset information
df.info()
df.columns
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16150 entries, 0 to 16149
Data columns (total 36 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       16150 non-null  int64  
 1   user_id          16150 non-null  int64  
 2   phq1             15864 non-null  float64
 3   phq2             15613 non-null  float64
 4   phq3             15613 non-null  float64
 5   phq4             15613 non-null  float64
 6   phq5             15613 non-null  float64
 7   phq6             15613 non-null  float64
 8   phq7             15613 non-null  float64
 9   phq8             15613 non-null  float64
 10  phq9             15613 non-null  float64
 11  age              9694 non-null   float64
 12  sex              14840 non-null  object 
 13  q1               1973 non-null   float64
 14  q2               1763 non-null   float64
 15  q3               1924 non-null   float64
 16  q4               1713 non-null   float64
 17  q5          

(16150, 36)

In [4]:
missing_values = df.isnull().sum()
missing_values

Unnamed: 0             0
user_id                0
phq1                 286
phq2                 537
phq3                 537
phq4                 537
phq5                 537
phq6                 537
phq7                 537
phq8                 537
phq9                 537
age                 6456
sex                 1310
q1                 14177
q2                 14387
q3                 14226
q4                 14437
q5                 14159
q6                 14331
q7                 14414
q8                 14260
q9                 14258
q10                14307
q11                14348
q12                14374
q13                12959
q14                12309
q16                14226
q46                14343
q47                14143
happiness.score        0
time                   0
period.name            0
start.time             0
phq.day                0
id                     0
dtype: int64

In [5]:
missing_values[missing_values > 0].sort_values(ascending=False)

q4      14437
q7      14414
q2      14387
q12     14374
q11     14348
q46     14343
q6      14331
q10     14307
q8      14260
q9      14258
q16     14226
q3      14226
q1      14177
q5      14159
q47     14143
q13     12959
q14     12309
age      6456
sex      1310
phq2      537
phq9      537
phq8      537
phq7      537
phq6      537
phq5      537
phq4      537
phq3      537
phq1      286
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df = df.drop_duplicates()

In [8]:
df.shape

(16150, 36)

In [9]:
phq_columns = ["phq1", "phq2", "phq3", "phq4", "phq5", "phq6", "phq7", "phq8", "phq9"]

df[phq_columns].head()

,phq1,phq2,phq3,phq4,phq5,phq6,phq7,phq8,phq9
0,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,3.0
1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,3.0
2,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,3.0
3,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,3.0
4,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,3.0


In [10]:
df = df.dropna(subset=phq_columns).copy()

In [11]:
df["phq9_score"] = df[phq_columns].sum(axis=1)

df[["phq9_score"]].head()

,phq9_score
0,23.0
1,23.0
2,23.0
3,23.0
4,23.0


In [12]:
df.shape

(15613, 37)

In [13]:
def depression_severity(score):
    if score <= 4:
        return "Minimal"
    elif score <= 9:
        return "Mild"
    elif score <= 14:
        return "Moderate"
    elif score <= 19:
        return "Moderately Severe"
    else:
        return "Severe"

df["Severity"] = df["phq9_score"].apply(depression_severity)

df[["phq9_score", "Severity"]].head()

,phq9_score,Severity
0,23.0,Severe
1,23.0,Severe
2,23.0,Severe
3,23.0,Severe
4,23.0,Severe


In [14]:
df["phq9_score"].describe()

count    15613.000000
mean        15.841991
std          6.433538
min          3.000000
25%         11.000000
50%         17.000000
75%         21.000000
max         27.000000
Name: phq9_score, dtype: float64

In [15]:
df["Severity"].value_counts()

Severity
Severe               5057
Moderately Severe    4062
Moderate             3405
Mild                 2204
Minimal               885
Name: count, dtype: int64

In [16]:
import plotly.express as px

In [17]:
severity_counts = df["Severity"].value_counts().reset_index()
severity_counts.columns = ["Severity", "Count"]

fig = px.bar(
    severity_counts,
    x="Severity",
    y="Count",
    title="Distribution of Depression Severity",
    labels={"Severity": "Depression Severity", "Count": "Number of Records"}
)
fig.update_layout(
    width=800,
    height=500
)

fig.show()

In [18]:
fig = px.histogram(
    df,
    x="phq9_score",
    nbins=20,
    title="Distribution of PHQ-9 Scores",
    labels={"phq9_score": "PHQ-9 Score"}
)
fig.update_layout(
    width=800,
    height=500
)
fig.show()

In [19]:
df["age"].isnull().sum()

np.int64(5919)

In [20]:
fig = px.histogram(
    df.dropna(subset=["age"]),
    x="age",
    nbins=20,
    title="Age Distribution of Participants",
    labels={"age": "Age"}
)

fig.update_layout(
    width=800,
    height=500
)

fig.show()

In [21]:
df["sex"].value_counts()

sex
female         10882
male            3868
transgender       90
Name: count, dtype: int64

In [22]:
sex_counts = df["sex"].value_counts().reset_index()
sex_counts.columns = ["Sex", "Count"]

fig = px.bar(
    sex_counts,
    x="Sex",
    y="Count",
    title="Gender Distribution of Participants",
    labels={"Sex": "Sex", "Count": "Number of Participants"}
)

fig.update_layout(
    width=700,
    height=450
)

fig.show()

In [23]:
severity_counts = df["Severity"].value_counts().reset_index()
severity_counts.columns = ["Severity", "Count"]

fig = px.pie(
    severity_counts,
    names="Severity",
    values="Count",
    title="Percentage Distribution of Depression Severity"
)

fig.update_layout(
    width=700,
    height=500
)

fig.show()

In [24]:
fig = px.box(
    df.dropna(subset=["sex"]),
    x="sex",
    y="phq9_score",
    title="PHQ-9 Score Distribution by Gender",
    labels={"sex": "Sex", "phq9_score": "PHQ-9 Score"}
)

fig.update_layout(
    width=800,
    height=500
)

fig.show()

In [25]:
fig = px.scatter(
    df.dropna(subset=["age"]),
    x="age",
    y="phq9_score",
    color="Severity",
    title="Relationship Between Age and PHQ-9 Score",
    labels={"age": "Age", "phq9_score": "PHQ-9 Score"}
)

fig.update_layout(
    width=900,
    height=500
)

fig.show()

In [26]:
numeric_df = df.select_dtypes(include=["int64", "float64"])

numeric_df.head()

,Unnamed: 0,user_id,phq1,phq2,phq3,phq4,phq5,phq6,phq7,phq8,...,q12,q13,q14,q16,q46,q47,happiness.score,phq.day,id,phq9_score
0,1,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,NaN,NaN,0.0,NaN,NaN,NaN,2,-0.951921,1,23.0
1,2,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,NaN,NaN,NaN,NaN,1.0,NaN,3,16.159965,1,23.0
2,3,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,NaN,4.0,NaN,NaN,NaN,NaN,3,15.540289,1,23.0
3,4,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,NaN,NaN,NaN,1.0,NaN,NaN,2,-1.209259,1,23.0
4,5,1,3.0,3.0,3.0,3.0,2.0,3.0,1.0,2.0,...,NaN,NaN,1.0,NaN,1.0,NaN,2,-2.142604,1,23.0


In [27]:
correlation_matrix = numeric_df.corr()

correlation_matrix

,Unnamed: 0,user_id,phq1,phq2,phq3,phq4,phq5,phq6,phq7,phq8,...,q12,q13,q14,q16,q46,q47,happiness.score,phq.day,id,phq9_score
Unnamed: 0,1.000000,0.997176,-0.124092,-0.011768,-0.084838,0.030872,-0.145392,-0.073325,-0.162728,-0.136656,...,-0.202679,-0.037423,-0.064180,-0.200934,-0.185294,-0.129990,-0.005419,0.073463,0.997176,-0.140346
user_id,0.997176,1.000000,-0.110996,-0.014594,-0.082108,0.036250,-0.144305,-0.080962,-0.171214,-0.133711,...,-0.206827,-0.034536,-0.067361,-0.201716,-0.183639,-0.130943,-0.004868,0.073662,1.000000,-0.138851
phq1,-0.124092,-0.110996,1.000000,0.531737,0.271785,0.442861,0.295723,0.320930,0.274065,0.304486,...,0.332106,0.255678,0.191483,0.252232,0.349255,0.301030,-0.336877,-0.114857,-0.110996,0.597673
phq2,-0.011768,-0.014594,0.531737,1.000000,0.336582,0.556861,0.426760,0.689672,0.487210,0.359412,...,0.319125,0.272275,0.288921,0.291910,0.479584,0.479453,-0.424081,0.059852,-0.014594,0.795550
phq3,-0.084838,-0.082108,0.271785,0.336582,1.000000,0.597254,0.434126,0.315154,0.410195,0.314081,...,0.263782,0.298654,0.203171,0.005331,0.234604,0.219537,-0.148375,-0.040792,-0.082108,0.604024
phq4,0.030872,0.036250,0.442861,0.556861,0.597254,1.000000,0.493013,0.476331,0.454897,0.360308,...,0.375232,0.259776,0.225689,0.148045,0.328071,0.306353,-0.332598,-0.028171,0.036250,0.739582
phq5,-0.145392,-0.144305,0.295723,0.426760,0.434126,0.493013,1.000000,0.458153,0.473919,0.367230,...,0.250606,0.192671,0.293266,0.120557,0.257446,0.223681,-0.169747,-0.075414,-0.144305,0.682947
phq6,-0.073325,-0.080962,0.320930,0.689672,0.315154,0.476331,0.458153,1.000000,0.527765,0.386772,...,0.332727,0.275284,0.298345,0.296692,0.484712,0.478622,-0.365620,0.032918,-0.080962,0.771419
phq7,-0.162728,-0.171214,0.274065,0.487210,0.410195,0.454897,0.473919,0.527765,1.000000,0.459797,...,0.463791,0.181592,0.253095,0.210799,0.346587,0.346735,-0.266842,-0.012962,-0.171214,0.720727
phq8,-0.136656,-0.133711,0.304486,0.359412,0.314081,0.360308,0.367230,0.386772,0.459797,1.000000,...,0.289182,0.198652,0.310374,0.282391,0.244491,0.205260,-0.230143,-0.085233,-0.133711,0.638981


In [28]:
fig = px.imshow(
    correlation_matrix,
    title="Correlation Heatmap of Numeric Features",
    text_auto=False,
    aspect="auto"
)

fig.update_layout(
    width=1000,
    height=800
)

fig.show()

In [29]:
correlation_with_phq9 = correlation_matrix["phq9_score"].sort_values(ascending=False)

correlation_with_phq9

phq9_score         1.000000
phq2               0.795550
phq6               0.771419
phq4               0.739582
phq7               0.720727
phq5               0.682947
phq8               0.638981
phq9               0.607317
q10                0.606484
phq3               0.604024
phq1               0.597673
q7                 0.523885
q8                 0.518113
q46                0.513992
q6                 0.491120
q9                 0.489378
q3                 0.489056
q47                0.482534
q2                 0.460583
q4                 0.457340
q12                0.456525
q1                 0.450881
q5                 0.437853
q11                0.428921
q14                0.384593
q16                0.369787
q13                0.340853
phq.day           -0.034597
user_id           -0.138851
id                -0.138851
Unnamed: 0        -0.140346
age               -0.251608
happiness.score   -0.422905
Name: phq9_score, dtype: float64

In [30]:
severity_order = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]

avg_phq_by_severity = (
    df.groupby("Severity")["phq9_score"]
    .mean()
    .reindex(severity_order)
    .reset_index()
)

avg_phq_by_severity

,Severity,phq9_score
0,Minimal,3.822599
1,Mild,6.903811
2,Moderate,12.302496
3,Moderately Severe,17.494830
4,Severe,22.896579


In [31]:
fig = px.bar(
    avg_phq_by_severity,
    x="Severity",
    y="phq9_score",
    title="Average PHQ-9 Score by Depression Severity",
    labels={
        "Severity": "Depression Severity",
        "phq9_score": "Average PHQ-9 Score"
    }
)

fig.update_layout(
    width=850,
    height=500
)

fig.show()

In [32]:
drop_columns = ["Severity", "phq9_score"] + phq_columns

X = df.drop(drop_columns, axis=1)
y = df["Severity"]

X.head()

,Unnamed: 0,user_id,age,sex,q1,q2,q3,q4,q5,q6,...,q14,q16,q46,q47,happiness.score,time,period.name,start.time,phq.day,id
0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,2,2017-01-22 20:11:59,evening,2017-01-09 07:22:37,-0.951921,1
1,2,1,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,...,NaN,NaN,1.0,NaN,3,2017-02-08 22:53:06,evening,2017-01-09 07:22:37,16.159965,1
2,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3,2017-02-08 08:00:46,morning,2017-01-09 07:22:37,15.540289,1
3,4,1,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,NaN,NaN,2,2017-01-22 14:01:25,midday,2017-01-09 07:22:37,-1.209259,1
4,5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,NaN,1.0,NaN,2,2017-01-21 15:37:24,midday,2017-01-09 07:22:37,-2.142604,1


In [33]:
X.shape, y.shape


((15613, 27), (15613,))

In [34]:
columns_to_remove = ["Unnamed: 0", "user_id", "id", "time", "start.time"]

X = X.drop(columns=columns_to_remove, axis=1, errors="ignore")

X.head()


,age,sex,q1,q2,q3,q4,q5,q6,q7,q8,...,q11,q12,q13,q14,q16,q46,q47,happiness.score,period.name,phq.day
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,2,evening,-0.951921
1,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,...,4.0,NaN,NaN,NaN,NaN,1.0,NaN,3,evening,16.159965
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,NaN,NaN,3,morning,15.540289
3,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,2,midday,-1.209259
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,1.0,NaN,2,midday,-2.142604


In [35]:
X.columns

Index(['age', 'sex', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9',
       'q10', 'q11', 'q12', 'q13', 'q14', 'q16', 'q46', 'q47',
       'happiness.score', 'period.name', 'phq.day'],
      dtype='object')

In [36]:
X.shape


(15613, 22)

In [37]:
# Separate numeric and categorical columns
numeric_columns = X.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = X.select_dtypes(include=["object"]).columns

# Fill missing numeric values with median
X[numeric_columns] = X[numeric_columns].fillna(X[numeric_columns].median())

# Fill missing categorical values with mode
for col in categorical_columns:
    X[col] = X[col].fillna(X[col].mode()[0])

# Check missing values again
X.isnull().sum().sort_values(ascending=False).head(20)

age                0
sex                0
period.name        0
happiness.score    0
q47                0
q46                0
q16                0
q14                0
q13                0
q12                0
q11                0
q10                0
q9                 0
q8                 0
q7                 0
q6                 0
q5                 0
q4                 0
q3                 0
q2                 0
dtype: int64

In [38]:
# Encode categorical columns
label_encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

X.head()

,age,sex,q1,q2,q3,q4,q5,q6,q7,q8,...,q11,q12,q13,q14,q16,q46,q47,happiness.score,period.name,phq.day
0,29.0,0,2.0,1.0,3.0,2.0,3.0,3.0,2.0,3.0,...,2.0,3.0,2.0,0.0,2.0,3.0,3.0,2,0,-0.951921
1,29.0,0,2.0,1.0,3.0,2.0,4.0,3.0,2.0,3.0,...,4.0,3.0,2.0,0.0,2.0,1.0,3.0,3,0,16.159965
2,29.0,0,2.0,1.0,3.0,2.0,3.0,3.0,2.0,3.0,...,2.0,3.0,4.0,0.0,2.0,3.0,3.0,3,2,15.540289
3,29.0,0,4.0,1.0,3.0,2.0,3.0,3.0,3.0,3.0,...,2.0,3.0,2.0,0.0,1.0,3.0,3.0,2,1,-1.209259
4,29.0,0,2.0,1.0,3.0,2.0,3.0,3.0,2.0,3.0,...,2.0,3.0,2.0,1.0,2.0,1.0,3.0,2,1,-2.142604


In [39]:
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

target_encoder.classes_


array(['Mild', 'Minimal', 'Moderate', 'Moderately Severe', 'Severe'],
      dtype=object)

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape


((12490, 22), (3123, 22), (12490,), (3123,))

In [41]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

((12490, 22), (3123, 22))

In [42]:
# Create the two models
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
random_forest_model = RandomForestClassifier(random_state=42)

# Train Logistic Regression
logistic_model.fit(X_train_scaled, y_train)

# Train Random Forest
random_forest_model.fit(X_train_scaled, y_train)

print("Both models trained successfully.")

Both models trained successfully.


In [43]:
# Make predictions
logistic_predictions = logistic_model.predict(X_test_scaled)
random_forest_predictions = random_forest_model.predict(X_test_scaled)

# Calculate accuracy
logistic_accuracy = accuracy_score(y_test, logistic_predictions)
random_forest_accuracy = accuracy_score(y_test, random_forest_predictions)

print("Logistic Regression Accuracy:", logistic_accuracy)
print("Random Forest Accuracy:", random_forest_accuracy)


Logistic Regression Accuracy: 0.4543707973102786
Random Forest Accuracy: 0.7108549471661864


In [44]:
model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [logistic_accuracy, random_forest_accuracy]
})

model_results

,Model,Accuracy
0,Logistic Regression,0.454371
1,Random Forest,0.710855


In [45]:
fig = px.bar(
    model_results,
    x="Model",
    y="Accuracy",
    title="Model Accuracy Comparison",
    labels={"Model": "Machine Learning Model", "Accuracy": "Accuracy"}
)

fig.update_layout(
    width=700,
    height=450
)

fig.show()

In [46]:
# For regression, target should be numeric PHQ-9 score
y_regression = df["phq9_score"]

# Use same feature columns as before
X_regression = X.copy()

X_regression.shape, y_regression.shape

((15613, 22), (15613,))

In [47]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_regression,
    y_regression,
    test_size=0.2,
    random_state=42
)

X_train_reg.shape, X_test_reg.shape, y_train_reg.shape, y_test_reg.shape

((12490, 22), (3123, 22), (12490,), (3123,))

In [48]:
scaler_reg = StandardScaler()

X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

X_train_reg_scaled.shape, X_test_reg_scaled.shape

((12490, 22), (3123, 22))

In [49]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [50]:
linear_regression_model = LinearRegression()
random_forest_regression_model = RandomForestRegressor(random_state=42)

linear_regression_model.fit(X_train_reg_scaled, y_train_reg)
random_forest_regression_model.fit(X_train_reg_scaled, y_train_reg)

print("Both regression models trained successfully.")

Both regression models trained successfully.


In [51]:
from sklearn.metrics import r2_score, mean_squared_error

In [52]:
# Predictions
linear_predictions = linear_regression_model.predict(X_test_reg_scaled)
rf_regression_predictions = random_forest_regression_model.predict(X_test_reg_scaled)

# Metrics
linear_r2 = r2_score(y_test_reg, linear_predictions)
linear_mse = mean_squared_error(y_test_reg, linear_predictions)
linear_rmse = np.sqrt(linear_mse)

rf_r2 = r2_score(y_test_reg, rf_regression_predictions)
rf_mse = mean_squared_error(y_test_reg, rf_regression_predictions)
rf_rmse = np.sqrt(rf_mse)

print("Linear Regression")
print("R2 Score:", linear_r2)
print("MSE:", linear_mse)
print("RMSE:", linear_rmse)

print("\nRandom Forest Regressor")
print("R2 Score:", rf_r2)
print("MSE:", rf_mse)
print("RMSE:", rf_rmse)

Linear Regression
R2 Score: 0.3036996741915522
MSE: 29.049421288059666
RMSE: 5.389751505223563

Random Forest Regressor
R2 Score: 0.6174988543206121
MSE: 15.957822382324688
RMSE: 3.994724318688924


In [53]:
regression_results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor"],
    "R2 Score": [linear_r2, rf_r2],
    "MSE": [linear_mse, rf_mse],
    "RMSE": [linear_rmse, rf_rmse]
})

regression_results

,Model,R2 Score,MSE,RMSE
0,Linear Regression,0.303700,29.049421,5.389752
1,Random Forest Regressor,0.617499,15.957822,3.994724


In [ ]:
import pickle
import os

save_path = "/Users/grishma/Desktop/test"

with open(os.path.join(save_path, "random_forest_model.pkl"), "wb") as file:
    pickle.dump(random_forest_model, file)

with open(os.path.join(save_path, "scaler.pkl"), "wb") as file:
    pickle.dump(scaler, file)

with open(os.path.join(save_path, "target_encoder.pkl"), "wb") as file:
    pickle.dump(target_encoder, file)

with open(os.path.join(save_path, "feature_columns.pkl"), "wb") as file:
    pickle.dump(X.columns.tolist(), file)

print("Files saved successfully in:", save_path)